# Arm L — Qwen3-VL-8B, full-stack E2E harness (GPU, Colab-only)

Local-model counterpart to Arm P (`src/e2e_harness/poc_run_arm_p_v2.py`, GPT-5.5-low).
Same real production code path, same holdout, same scoring — only the Stage 4
symbol-detection VLM call is swapped from GPT-5.5 to **Qwen3-VL-8B running on a Colab GPU**.

Runs, per holdout sheet:

```
real system prompt + tool schema  ->  PaddleOCR (stage 1.5 sub)  ->  real tile grid
  ->  Qwen3-VL per tile (JSON-in-prompt, no native tool-calling)  ->  real NMS/compose
  ->  real stage_06_run (line tracing)  ->  real detections_to_entities / build_relations
  ->  score vs PID2Graph ground truth (graph_matcher.py)
```

**This CANNOT run locally** — there's no GPU on the dev machine worth using for an 8B VLM.
This notebook is Colab-only, following this project's established GPU-only notebook pattern
(e.g. `Stage105_SkidMatrix_Molmo2_Qwen_Adapters_GPUOnly.ipynb`). No Google Drive is used
anywhere — all shared storage (model/adapter weights, datasets, and this project's own
private code) goes through Hugging Face, per this project's standing rule.

## Deliberate exception to the CPU/GPU split (H6)

This project's standing rule is "new notebooks split CPU-prep (local + HF) from GPU-only
Colab cells." **Cascade mode here breaks that rule on purpose.** Per `E2E_Harness_Plan.md`
§H6: *"Cascade-mode Arm L must run GPU inference AND the agent's deterministic CPU code
(tiling, line tracing, graph construction, the conversion layer) interleaved per
stage-transition within ONE Colab session, because stage N's real input depends on stage
N-1's real output for that SAME arm — there's no way to pre-batch that."* Concretely: Stage
4 (Qwen, GPU) produces detections that Stage 6 (real Python line-tracer, CPU) consumes,
whose output `detections_to_entities`/`build_relations` (CPU) consume, before anything can
be scored. All of that has to live in one process holding the GPU model in memory. This is
the one place in this project that pattern is broken, and it's broken for this documented
reason, not by accident.

## Decisions made while building this (read before running)

1. **Model config: Qwen3-VL-8B BASE, no LoRA adapter, for the detection call.**
   No prior benchmark in this repo ever validated a Qwen adapter *for detection
   specifically* — `Stage4_Detection_GPT55_vs_Molmo2.ipynb` only ever compared GPT-5.5 vs
   Molmo2 for Stage 4 (Molmo2 zero-shot F1 = 0.434, never a Qwen row). Where adapters
   *were* tested on tasks other than what they were trained for, they hurt, sometimes
   badly: `results.csv` (`stage105_synthskid_qwen_matrix`) shows `qwen+v2-general` and
   `qwen+v3-relation` produced **12/12 parse failures** on the skid-grouping task (unparseable
   output, not just wrong answers), and `qwen+v3-stage13` scored *below* the trivial
   baseline on 2/3 prompts — real negative transfer. Given that evidence, attaching any
   existing adapter to an unvalidated 4th task (detection) would be a guess dressed as a
   choice. Base model, zero-shot, is the honest starting point; if it's promising, training
   a dedicated `v4-detection` adapter is a well-motivated next step, not this notebook's job.
2. **JSON-in-prompt, not native tool-calling.** Qwen3-VL (via `transformers`
   `AutoModelForImageTextToText.generate()`) has no Anthropic/OpenAI-style function-calling.
   The real `detect_symbols` Anthropic tool schema (`ontology_render.build_detection_tool_schema`)
   is translated into an explicit "respond with exactly one fenced ```json block matching
   this shape" instruction appended to the same real system prompt, and the one line of
   `prompt.py` that says *"You MUST call the `detect_symbols` tool. Never respond with free
   text"* is substituted for a Qwen-appropriate equivalent (same substitution pattern this
   project already uses for the `connector`→`inlet_outlet` ontology-name fix in
   `poc_run_arm_p_v2.py` — editing reused prod text via `.replace()`, never editing
   `prompt.py` itself). Parsing reuses this project's own established fenced-JSON extraction
   convention (`e2e_bench/backends/parse_json_common.py`'s `_extract_json_text` — last
   fenced ```json block wins, then balanced-bracket fallback), the same one already used for
   Qwen elsewhere in this project (titleblock/skid/entity/relation stages). **Honesty
   caveat: whether Qwen3-VL reliably produces parseable, schema-shaped JSON for THIS
   specific multi-array, many-field detection schema (much richer than the single JSON
   array `[x0,y0,x1,y1,confidence,"entity_type"]` shape used in the existing Qwen-vs-GPT
   detection notebook) is UNVERIFIED — I could not run this. Per-tile parse-failure rate is
   tracked and printed precisely so this is visible, not papered over, the first time this
   runs.**
3. **Private code transfer: zip + private HF dataset repo, not `pip install git+https`.**
   The real agent (`pnid_agent`) and its monorepo dependencies (`entity_operations`,
   `rive_adk`, `rive_security`) live in a **private** repo
   (`rive-labs/rive-ai-platform`, confirmed via `.venv-e2e`'s `pip freeze`:
   `-e git+https://github.com/rive-labs/rive-ai-platform.git@<sha>#egg=...`). Colab has no
   credentials for that private git remote. `pid-ml` itself IS mirrored to a public GitHub
   repo, but `src/e2e_bench/` and `src/e2e_harness/` are **untracked** locally
   (`git status --short src/` shows `??`) — not on the public mirror either. Rather than
   inventing a new transfer mechanism, this reuses the project's own established pattern
   (private HF repo, `hf_hub_download`/`snapshot_download` + local install, same as the
   adapter weights) for BOTH pieces at once: `scripts/package_agent_src_for_colab.sh` zips
   all of it (private monorepo code + pid-ml's own untracked `e2e_bench`/`e2e_harness`) and
   pushes to a **private** HF dataset repo. **This script does not run itself — it must be
   run locally, deliberately, by you** (see run order at the bottom of this notebook).
4. **No hard OS/platform blockers found.** Checked every `pyproject.toml` in the dependency
   chain (`pnid-intelligence-agent`, `entity_operations`, `rive_adk`, `security`) — all
   dependencies are ordinary cross-platform PyPI packages (FastAPI, pydantic, httpx,
   sqlalchemy, opencv-python, scikit-image, pymupdf, neo4j/kafka/redis clients, etc.), no
   macOS-only or compiled-on-this-Mac artifacts. The one system-level dependency is
   `python-magic`, which needs `libmagic1` at the OS level — trivial via `apt-get` on
   Colab's Ubuntu runtime (included in the install cell below). `rive_adk` declares
   `rive-security>=0.1.0`, which **does not exist on PyPI** — it's only satisfiable by
   installing the editable local copy first, so install order matters (security →
   entity_operations → rive_adk → pnid_agent), same order `.venv-e2e` used.
5. **OCR: real PaddleOCR, in-notebook, CPU.** Same substitute Arm P v2 uses (prod's real
   Stage 1.5 is Google Cloud Vision; no API key available in this project). PaddleOCR runs
   fine as a plain CPU pip install in Colab — no GPU needed for it, so it does not fight the
   VLM for VRAM.
6. **Ground truth / holdout: `src/e2e_harness/e2e_holdout_ids.json`.** Starts with just
   `PID2Graph OPEN100 / sheet 8` (same sheet Arm P v2/v3 were scored on, for direct
   comparison) — the last cell shows how to loop the same function over all 4 frozen holdout
   sheets if time permits.


## 1. Config

In [ ]:
# ── Model / adapter config ──────────────────────────────────────────────────
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
USE_ADAPTER = None   # deliberately None for detection — see notebook intro, decision (1)

# ── HF (datasets, private agent-code zip). No Google Drive, ever. ──────────
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"
DATA_REPO = "timthy45/pnid-extraction-datasets"       # public-within-org dataset repo, holds PID2Graph.zip etc (established repo, used by earlier notebooks in this project)
AGENT_SRC_REPO = "timthy45/pnid-agent-src"             # PRIVATE dataset repo — created by scripts/package_agent_src_for_colab.sh (run LOCALLY first, see bottom of this notebook)
AGENT_SRC_FILE = "agent_src/latest.zip"                # stable pointer the packaging script always overwrites

# ── Holdout sheet ────────────────────────────────────────────────────────────
# One of the 4 frozen sheets in e2e_holdout_ids.json. Starting with the same sheet
# Arm P v2/v3 were scored on for direct comparability. See section 11 to loop all 4.
HOLDOUT_TREE = "PID2Graph OPEN100"
HOLDOUT_STEM = "8"

assert HF_TOKEN.startswith("hf_") and HF_TOKEN != "PASTE_YOUR_HF_TOKEN_HERE", "paste your HF token"


## 2. Install (GPU-runtime prep — this cell still needs the GPU runtime type set,
even though it's pip installs, because everything downstream in this notebook runs
in the SAME process per decision/H6 above — there's no separate CPU-only notebook here)

`transformers==4.57.1` pinned — same version this project's other Qwen3-VL notebooks use
(known-good for `Qwen3-VL-8B-Instruct` + `AutoModelForImageTextToText`/`peft`).
`libmagic1` via apt — the one system-level (non-pip) dependency in the whole chain
(`python-magic`, pulled in transitively by `pnid_agent`'s Stage 0 ingestion code).

In [ ]:
!apt-get -qq update && apt-get -qq install -y libmagic1 > /dev/null
!pip install -q transformers==4.57.1 accelerate peft huggingface_hub
!pip install -q paddleocr paddlepaddle
!pip install -q pymupdf python-magic

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")


## 3. Private code: pnid_agent + its monorepo deps + this repo's own e2e_bench/e2e_harness

Downloads the zip `scripts/package_agent_src_for_colab.sh` builds and pushes (see run order
at the bottom). Installs the 4 private editable packages **in dependency order**
(`rive_security` first — `rive_adk` needs it and it doesn't exist on PyPI — then
`entity_operations`, then `rive_adk`, then `pnid_agent` itself), then puts this repo's own
`e2e_bench`/`e2e_harness` (not proper pip packages, just `sys.path`-imported like
`poc_run_arm_p_v2.py` does locally) on `sys.path`.

In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

AGENT_SRC_ROOT = Path("/content/agent_src")
AGENT_SRC_ROOT.mkdir(exist_ok=True)

zp = hf_hub_download(repo_id=AGENT_SRC_REPO, filename=AGENT_SRC_FILE,
                      repo_type="dataset", token=HF_TOKEN)
with zipfile.ZipFile(zp) as zf:
    zf.extractall(AGENT_SRC_ROOT)
print("extracted:", sorted(p.name for p in AGENT_SRC_ROOT.iterdir()))


In [ ]:
# Install order matters: rive_security -> entity_operations -> rive_adk -> pnid_agent.
# (rive_adk's pyproject declares rive-security>=0.1.0, which is NOT on PyPI — only the
# editable local copy satisfies it, same as .venv-e2e's install order on the Mac.)
!pip install -q -e /content/agent_src/shared/security
!pip install -q -e /content/agent_src/shared/entity_operations
!pip install -q -e /content/agent_src/shared/rive_adk
!pip install -q -e "/content/agent_src/agents/pnid-intelligence-agent"

import sys
sys.path.insert(0, "/content/agent_src/pid_ml_src")
print("sys.path updated for e2e_bench / e2e_harness")


In [ ]:
# Fail fast with a clear message rather than a confusing import error later.
import importlib

REQUIRED_MODULES = [
    "pnid_agent.models.page_classification", "pnid_agent.models.page_ocr",
    "pnid_agent.models.detections", "pnid_agent.models.line_tracing",
    "pnid_agent.models.drawing_document", "pnid_agent.shared.coord_ops",
    "pnid_agent.stages.tile_segmentation.grid",
    "pnid_agent.sub_agents.symbol_detection.nms",
    "pnid_agent.sub_agents.symbol_detection.driver",
    "pnid_agent.sub_agents.symbol_detection.prompt",
    "pnid_agent.sub_agents.symbol_detection.ontology_render",
    "pnid_agent.sub_agents.symbol_detection.tile_words",
    "pnid_agent.stages.line_tracing.driver",
    "pnid_agent.stages.graph_construction.relations",
    "e2e_bench.assembly.document", "e2e_bench.assembly.entities",
    "e2e_bench.converters.stage01_classification", "e2e_bench.converters.stage04_detection",
    "e2e_bench.ontology", "e2e_bench.types",
    "e2e_harness.graph_matcher", "e2e_harness.ground_truth", "e2e_harness.holdout",
]
missing = []
for mod in REQUIRED_MODULES:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing.append(f"{mod}: {type(e).__name__}: {e}")
if missing:
    raise RuntimeError("Missing/broken imports:\n  " + "\n  ".join(missing))
print(f"all {len(REQUIRED_MODULES)} required modules import cleanly")


## 4. Ground-truth data — PID2Graph (same zip earlier notebooks in this project already use)

Downloads `pid2graph/PID2Graph.zip` from the shared `DATA_REPO` and extracts it. The zip
contains both the `Complete` tree (raw sheet PNG + graphml — what `e2e_harness/holdout.py`
expects) and the `Patched` tree (patch-level crops used for adapter training) — both are
present; this notebook only needs `Complete`.

`e2e_harness/holdout.py`'s `_PID2GRAPH_COMPLETE_ROOT` is hardcoded to a local scratchpad path
that only exists on the Mac this was built on — **not portable to Colab**. This cell monkeypatches
it to the Colab-local extraction path instead of editing the source file.

In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data"); DATA.mkdir(exist_ok=True)
zp = hf_hub_download(repo_id=DATA_REPO, filename="pid2graph/PID2Graph.zip",
                      repo_type="dataset", token=HF_TOKEN)
pid2graph_dir = DATA / "pid2graph"
if not (pid2graph_dir / "PID2Graph" / "Complete").exists():
    pid2graph_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(pid2graph_dir)

COMPLETE_ROOT = str(pid2graph_dir / "PID2Graph" / "Complete")
assert Path(COMPLETE_ROOT).exists(), COMPLETE_ROOT
print("PID2Graph Complete tree ready at", COMPLETE_ROOT)


In [ ]:
import e2e_harness.holdout as holdout_mod

# Monkeypatch the Mac-local scratchpad path to this Colab session's extraction path
# (see markdown above) — everything else in holdout.py is reused unmodified.
holdout_mod._PID2GRAPH_COMPLETE_ROOT = COMPLETE_ROOT

def resolve_sheet(tree: str, stem: str) -> dict:
    base = f"{COMPLETE_ROOT}/{tree}/{stem}"
    sheet_id = f"{tree.replace(' ', '')}_{stem}"
    return {"sheet_id": sheet_id, "graphml_path": f"{base}.graphml", "png_path": f"{base}.png"}

sheet = resolve_sheet(HOLDOUT_TREE, HOLDOUT_STEM)
for k, v in sheet.items():
    if k != "sheet_id":
        assert Path(v).exists(), v
print(sheet)


## 5. Load Qwen3-VL-8B (base, no adapter — decision (1) above)

Identical load recipe to `Stage105_SkidMatrix_Molmo2_Qwen_Adapters_GPUOnly.ipynb`'s Qwen
section: `AutoProcessor`/`AutoModelForImageTextToText`, `dtype=torch.bfloat16`,
`device_map="cuda"`. No `peft.PeftModel` wrapping — deliberately base-only per decision (1).
This is the moment cascade mode (H6) begins: the model stays resident in GPU memory for the
rest of this notebook while the CPU-side agent code below runs interleaved with it.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model = AutoModelForImageTextToText.from_pretrained(
    QWEN_MODEL_ID, dtype=torch.bfloat16, device_map="cuda").eval()
print("Qwen3-VL-8B (base) loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

if USE_ADAPTER:
    from peft import PeftModel
    from huggingface_hub import snapshot_download
    local = Path(f"/content/adp_{USE_ADAPTER.replace('/', '_')}")
    snapshot_download(repo_id="timthy45/qwen3vl-pnid-domain-base", repo_type="model",
                       token=HF_TOKEN, allow_patterns=[f"{USE_ADAPTER}/*"], local_dir=str(local))
    qwen_model = PeftModel.from_pretrained(qwen_model, str(local / USE_ADAPTER))
    qwen_model.eval()
    print(f"adapter attached: {USE_ADAPTER}  (NOT the default — see decision (1) in the intro)")

def qwen_generate(image, prompt_text, max_new_tokens=4096):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, {"type": "text", "text": prompt_text}]}]
    inputs = qwen_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(qwen_model.device)
    with torch.no_grad():
        out = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return qwen_processor.decode(gen, skip_special_tokens=True).strip()


## 6. Real production system prompt + tool schema, translated for JSON-in-prompt

Same real code Arm P uses: `render_pid_ontology_for_prompt` / `build_system_prompt` /
`build_detection_tool_schema` (`pnid_agent.sub_agents.symbol_detection.*`), same 7-entity-type
benchmark ontology (`e2e_bench.ontology.entity_type_names`), same `connector`→`inlet_outlet`
substitution Arm P needed (real ISA-reference text names an off-page-connector entity_type
that doesn't exist in this benchmark's 7-type ontology; substituting the reused prod text is
consistent with Arm P's own precedent — not touching `prompt.py` itself).

The ONE additional substitution Arm L needs that Arm P didn't: prod's prompt ends with
*"You MUST call the `detect_symbols` tool. Never respond with free text"* — meaningless for a
model with no tool-calling. Replaced with an equivalent fenced-JSON instruction, built
directly from the real tool schema's field definitions so the shape asked for is still the
real one, not an invented shape. **Caveat: the merged per-entity-type `attributes` sub-schema
is empty for this benchmark's ontology (`_EmptyAttrs`, same as Arm P) — if this were ever
extended to entity types carrying real attributes, this instruction text would need to render
those fields too, not just the fixed ones below.**

In [ ]:
import json
from pydantic import BaseModel

from e2e_bench.ontology import entity_type_names
from pnid_agent.sub_agents.symbol_detection.ontology_render import (
    build_detection_tool_schema, render_pid_ontology_for_prompt,
)
from pnid_agent.sub_agents.symbol_detection.prompt import build_system_prompt
from pnid_agent.sub_agents.title_block_extraction.ontology import EntityExtractionSchema


class _EmptyAttrs(BaseModel):
    pass


def build_benchmark_schemas():
    return [
        EntityExtractionSchema(entity_type=k, model=_EmptyAttrs, attribute_metadata={},
                                raw_sample_payload={"descriptions": {"en": v}})
        for k, v in entity_type_names().items()
    ]


TOOL_UNAVAILABLE_TEXT = "You MUST call the ``detect_symbols`` tool. Never respond with free text."

schemas = build_benchmark_schemas()
ontology_rendering = render_pid_ontology_for_prompt(schemas)
system_prompt = build_system_prompt(ontology_rendering)
system_prompt = system_prompt.replace('entity_type "connector"', 'entity_type "inlet_outlet"')
anthropic_tool_schema = build_detection_tool_schema(schemas)
entity_type_enum = anthropic_tool_schema["input_schema"]["properties"]["symbols"]["items"]["properties"]["entity_type"]["enum"]

JSON_INSTEAD_OF_TOOL_TEXT = f'''You have NO tool-calling ability. Instead, respond with EXACTLY
ONE fenced code block, opened with ```json and closed with ```, containing nothing else
outside the fence, and nothing inside the fence except one JSON object with this shape
(mirrors the detect_symbols tool schema described above):

{{
  "symbols": [
    {{"detection_id": "p0_t000_s00", "entity_type": one of {json.dumps(entity_type_enum)},
     "bbox_tile": [x0, y0, x1, y1], "confidence": 0.0-1.0,
     "value": "exact tag/label text if visible, else omit",
     "name": "human-readable name, else omit",
     "description": "one short sentence, else omit",
     "entity_subtype": "drawing-convention subtype, else omit"}}
  ],
  "associations": [
    {{"detection_id": "references a symbol above", "span_id": "p0_wN from the OCR list below, NEVER invent",
     "kind": one of ["tag_label","callout_label","line_label","service_label"], "confidence": 0.0-1.0}}
  ],
  "unmapped_observations": [
    {{"observation_id": "o0", "description": "what you saw and why it didn't match",
     "bbox_tile": [x0, y0, x1, y1], "confidence": 0.0-1.0}}
  ]
}}

symbols/associations/unmapped_observations must ALL be present, even if empty ([]).'''

system_prompt = system_prompt.replace(TOOL_UNAVAILABLE_TEXT, JSON_INSTEAD_OF_TOOL_TEXT)
assert JSON_INSTEAD_OF_TOOL_TEXT[:60] in system_prompt, "tool-call substitution didn't match — prompt.py text may have changed"
print(f"real system prompt: {len(system_prompt)} chars; entity_type enum: {entity_type_enum}")


## 7. PaddleOCR — stage 1.5 substitute (CPU, same as Arm P v2)

Verbatim reuse of Arm P v2's `run_paddle_ocr`/`serialize_tile_words` (same PaddleOCR 3.7.0
`.predict()` shape parse, same `[span_id] 'text' bbox=[...]` serialization the real prompt's
user message expects) — real prod Stage 1.5 is Google Cloud Vision (no key available in this
project); PaddleOCR is this project's own accepted local substitute, same tradeoff Arm P
already made. Runs on CPU — does not compete with Qwen for VRAM.

In [ ]:
from pnid_agent.models.page_ocr import OcrWord

def run_paddle_ocr(png_path: str):
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(lang="en")
    result = ocr.predict(png_path)
    if not result:
        return []
    page = result[0]
    texts = page.get("rec_texts", [])
    scores = page.get("rec_scores", [])
    boxes = page.get("rec_boxes", [])
    words = []
    for text, score, box in zip(texts, scores, boxes):
        bbox = [int(round(v)) for v in box]
        words.append(OcrWord(text=text, bbox=bbox, confidence=float(score)))
    return words


def serialize_tile_words(words_in_tile, *, page_index: int) -> str:
    if not words_in_tile:
        return "  (none)"
    lines = []
    for w in words_in_tile[:2500]:
        span_id = f"p{page_index}_w{w.global_idx}"
        lines.append(f"  [{span_id}] {w.text!r:40}  bbox={w.bbox}")
    return "\n".join(lines)


## 8. Real tiling + per-tile Qwen3-VL inference (CASCADE MODE — H6 in effect from here on)

`compute_tile_grid` (real prod tiling: 1024px tiles, 205px overlap) and
`slice_words_to_tile` (real per-tile OCR-word slicing, `margin_px=24` — same default Arm P
uses) run on CPU, interleaved per-tile with the GPU model call, because the same real
prod tiling code Arm P uses is reused unmodified here too — this is the interleaving H6
describes, not a design choice unique to this cell.

JSON parsing reuses this project's own established fenced-JSON extraction convention
(`_extract_json_text` from `e2e_bench/backends/parse_json_common.py` — last fenced ```json
block wins, then a balanced-bracket fallback) rather than inventing a new one. Per-tile
parse failures are counted, not silently swallowed — decision (2) in the intro flags this
as unverified without an actual run.

In [ ]:
from PIL import Image
from e2e_bench.backends.parse_json_common import _extract_json_text
from e2e_bench.types import NormalizedDetection
from pnid_agent.sub_agents.symbol_detection.tile_words import slice_words_to_tile
from pnid_agent.stages.tile_segmentation.grid import compute_tile_grid

Image.MAX_IMAGE_PIXELS = None
_valid_entity_types = set(entity_type_names().keys())
_parse_failures = 0
_parse_attempts = 0


def payload_to_detections(payload: dict) -> list:
    out = []
    for spec in (payload or {}).get("symbols") or []:
        if not isinstance(spec, dict):
            continue
        entity_type = spec.get("entity_type")
        bbox = spec.get("bbox_tile")
        if entity_type not in _valid_entity_types or not (isinstance(bbox, list) and len(bbox) == 4):
            continue
        value = spec.get("value") or spec.get("name") or None
        out.append(NormalizedDetection(
            bbox_tile=[float(v) for v in bbox], confidence=float(spec.get("confidence", 0.7)),
            entity_type=entity_type, value=value,
            entity_subtype=spec.get("entity_subtype"), description=spec.get("description"),
        ))
    return out


def qwen_detect_tile(tile_image, tile_id, tile_origin, words_in_tile, page_index):
    global _parse_failures, _parse_attempts
    _parse_attempts += 1
    words_dump = serialize_tile_words(words_in_tile, page_index=page_index)
    user_text = (
        f"Tile id: {tile_id}\n"
        f"Tile drawing-coord origin: x={tile_origin[0]}, y={tile_origin[1]} (top-left of the tile in PAGE coords).\n\n"
        "Stage 1.5 OCR words on (or near) this tile. Each line is "
        "``[span_id] 'text' bbox=[x0, y0, x1, y1]`` in ORIGINAL page coords. "
        "Use the FULL span_id verbatim in your associations:\n"
        f"{words_dump}\n\n"
        "Detect every symbol on the tile and associate tags by span_id. Return tile-local bboxes only. "
        "Remember: respond with ONLY the single fenced ```json block, nothing else."
    )
    full_prompt = system_prompt + "\n\n---\n\n" + user_text
    try:
        raw_text = qwen_generate(tile_image, full_prompt, max_new_tokens=4096)
        candidate = _extract_json_text(raw_text)
        if candidate is None:
            _parse_failures += 1
            print(f"  [parse-fail] tile {tile_id}: no JSON-like content in output")
            return {"symbols": [], "associations": [], "unmapped_observations": []}
        payload = json.loads(candidate)
        return payload
    except Exception as e:
        _parse_failures += 1
        print(f"  [parse-fail] tile {tile_id}: {type(e).__name__}: {e}")
        return {"symbols": [], "associations": [], "unmapped_observations": []}


In [ ]:
from e2e_bench.assembly.document import build_artifact_store, build_single_page_document
from e2e_bench.converters.stage01_classification import convert_classification
from e2e_bench.converters.stage04_detection import TileBatch, convert_detection
from e2e_bench.types import NormalizedWord
from pnid_agent.models.page_classification import PageClassificationLabel
import tempfile
from types import SimpleNamespace


async def run_arm_l(sheet_id: str, graphml_path: str, png_path: str) -> dict:
    global _parse_failures, _parse_attempts
    _parse_failures = _parse_attempts = 0
    print(f"=== Arm L (Qwen3-VL-8B base, real prompt+schema JSON-in-prompt, PaddleOCR) on {sheet_id} ===")

    store = build_artifact_store(tempfile.mkdtemp())
    doc = build_single_page_document(
        doc_id=sheet_id, job_id="job-armL-" + sheet_id, tenant_id="benchmark",
        image_path=png_path, artifact_store=store,
    )
    context = SimpleNamespace(tenant_id="benchmark")

    convert_classification(
        drawing_document=doc, artifact_store=store, page_index=0,
        classification=PageClassificationLabel.PID_DRAWING, confidence=1.0,
        model_version="qwen3vl-8b-base-assumed",
    )

    print("  running PaddleOCR (stage 1.5 substitute)...")
    page_words = run_paddle_ocr(png_path)
    print(f"  {len(page_words)} OCR words")

    full_img = Image.open(png_path).convert("RGB")
    W, H = full_img.size
    tiles = compute_tile_grid(drawing_bbox=[0, 0, W, H], page_size=(W, H))
    print(f"  {len(tiles)} tiles (1024/205 grid)")

    tile_batches = []
    for t in tiles:
        crop = full_img.crop((t.x0, t.y0, t.x1, t.y1))
        words_in_tile = slice_words_to_tile(page_words, tile_bbox=[t.x0, t.y0, t.x1, t.y1], margin_px=24)
        tile_id = f"p0_t{t.idx:03d}"
        payload = qwen_detect_tile(crop, tile_id, (t.x0, t.y0), words_in_tile, page_index=0)
        dets = payload_to_detections(payload)
        print(f"    tile {t.idx} ({t.x0},{t.y0})-({t.x1},{t.y1}): {len(words_in_tile)} ocr words, {len(dets)} detections")
        tile_batches.append(TileBatch(tile_index=t.idx, origin_xy=(t.x0, t.y0), upscale=1.0, detections=dets))

    parse_failure_rate = _parse_failures / max(_parse_attempts, 1)
    print(f"  tile parse-failure rate: {parse_failure_rate:.1%} ({_parse_failures}/{_parse_attempts})")

    normalized_ocr_words = [NormalizedWord(text=w.text, bbox=w.bbox, confidence=w.confidence) for w in page_words]

    s4_out, dropped = convert_detection(
        drawing_document=doc, artifact_store=store, page_index=0,
        tile_batches=tile_batches, ocr_words_for_page=normalized_ocr_words,
        model_version="qwen3vl-8b-base",
    )
    print(f"  stage4: {len(s4_out.pages[0].detections)} detections after NMS, {len(dropped)} dropped in compose")

    return await _finish_arm_l(sheet_id, graphml_path, doc, store, context, s4_out, W, H, parse_failure_rate, len(page_words), len(tiles))


## 9. Real stage 6 (line tracing) -> real entities/relations -> score

Identical to Arm P v2 from this point on — same `stage_06_run`, `detections_to_entities`,
`build_relations`, `parse_graphml_ground_truth`, `match_entities`/`match_relations`. This is
where cascade mode (H6) matters most: `stage_06_run` and everything after it is real,
deterministic Python that depends on Qwen's Stage 4 output for THIS sheet — it cannot be
pre-batched ahead of the GPU call.

In [ ]:
import asyncio

from e2e_bench.assembly.entities import detections_to_entities
from e2e_bench.ontology import load_ontology_relation_index
from e2e_harness.graph_matcher import match_entities, match_relations
from e2e_harness.ground_truth import equipment_only, parse_graphml_ground_truth
from pnid_agent.models.line_tracing import Stage06Output
from pnid_agent.stages.graph_construction.relations import build_relations
from pnid_agent.stages.line_tracing.driver import stage_06_run


async def _finish_arm_l(sheet_id, graphml_path, doc, store, context, s4_out, W, H,
                         parse_failure_rate, n_ocr_words, n_tiles):
    await stage_06_run(context, store, drawing_document=doc)
    s6_raw = store.read_json(doc.job_id, "stage-06/stage_06_output.json")
    s6_out = Stage06Output.model_validate(s6_raw)
    print(f"  stage6: {len(s6_out.pages[0].segments)} segments")

    entities, det_to_temp, type_by_temp = detections_to_entities(
        detections=s4_out.pages[0].detections, page_index=0, page_size=(W, H),
        stage_4_model_version="qwen3vl-8b-base",
    )
    print(f"  entities built: {len(entities)} (of {len(s4_out.pages[0].detections)} detections)")

    ontology_idx = load_ontology_relation_index()
    relations, _meta, unresolved = build_relations(s6_out.pages[0], det_to_temp, type_by_temp, ontology_idx)
    print(f"  relations built: {len(relations)}, unresolved: {len(unresolved)}")

    gt_entities, gt_edges = parse_graphml_ground_truth(graphml_path)
    gt_equip = equipment_only(gt_entities)
    print(f"  GT: {len(gt_equip)} equipment entities, {len(gt_edges)} edges")

    em = match_entities(entities, gt_equip)
    rm = match_relations(relations, gt_edges, em)

    result = {
        "sheet_id": sheet_id, "arm": "qwen3vl-8b-base-realprompt-jsoninprompt-paddleocr",
        "n_ocr_words": n_ocr_words, "n_tiles": n_tiles,
        "n_detections_post_nms": len(s4_out.pages[0].detections),
        "n_entities": len(entities), "n_relations": len(relations),
        "n_gt_entities": len(gt_equip), "n_gt_edges": len(gt_edges),
        "tile_parse_failure_rate": parse_failure_rate,
        "entity_precision": em.precision, "entity_recall": em.recall, "entity_f1": em.f1,
        "relation_precision": rm.precision, "relation_recall": rm.recall, "relation_f1": rm.f1,
    }
    print(json.dumps(result, indent=2))
    return result


### Async note

Colab's IPython kernel already runs an event loop, so `asyncio.run(...)` (what
`poc_run_arm_p_v2.py` uses as a plain script) doesn't work unmodified in a notebook cell.
`nest_asyncio` patches this — install once, then call `run_arm_l` with plain `await` (Colab
supports top-level `await` in a cell) or `asyncio.run`, either now works.

In [ ]:
!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()


## 10. Run on the primary holdout sheet

Same output shape as Arm P v2/v3's printed result — compare `entity_precision/recall/f1` and
`relation_precision/recall/f1` directly against Arm P's row in `results.csv`
(`e2e_armP_v2_gpt55low_realprompt_paddleocr`: entity F1 = 0.444, relation F1 = 0.0 on this
exact sheet).

In [ ]:
result = await run_arm_l(sheet["sheet_id"], sheet["graphml_path"], sheet["png_path"])


## 11. Optional — all 4 frozen holdout sheets

`src/e2e_harness/e2e_holdout_ids.json` freezes 4 sheets (2 PID2Graph OPEN100, 2 Dataset PID).
Loops the same `run_arm_l` over all 4 if time/VRAM budget allows — the model stays loaded,
only the per-sheet CPU-side work (OCR, tiling, stage 6, scoring) repeats.

In [ ]:
import e2e_harness.holdout as holdout_mod

RUN_ALL_4 = False   # flip to True to run the full holdout instead of just the primary sheet

if RUN_ALL_4:
    all_results = []
    for s in holdout_mod.load_holdout():
        sh = resolve_sheet(s["tree"], s["stem"])
        r = await run_arm_l(sh["sheet_id"], sh["graphml_path"], sh["png_path"])
        all_results.append(r)
    print(json.dumps(all_results, indent=2))


## 12. Push results to HF + free the GPU

Same "no MLflow, append to a flat results artifact" convention this project already uses
elsewhere in Colab (`Stage105_SkidMatrix...ipynb`'s summary-JSON push). Copy the printed
`entity_*`/`relation_*` numbers into `pid-ml/results.csv` by hand afterward (per this
project's `CLAUDE.md` schema), same as every other run in this repo.

In [ ]:
from huggingface_hub import HfApi

with open("/content/arm_l_result.json", "w") as f:
    json.dump({"primary_sheet": result, **({"all_4": all_results} if RUN_ALL_4 else {})}, f, indent=2)

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="/content/arm_l_result.json",
    path_in_repo=f"benchmarks/arm_l_qwen3vl8b_base_{sheet['sheet_id']}.json",
    repo_id=DATA_REPO, repo_type="dataset", token=HF_TOKEN)
print("results pushed to HF")

from google.colab import runtime
runtime.unassign()
